# BERT Sentiment Classification — Baseline vs. Class-Weighted (RQ2)

Runs `src/train.py` + `src/evaluate.py` twice — once with `use_class_weights: false` (baseline) and once with `use_class_weights: true` (RQ2 class-balancing experiment) — then compares the neutral-class F1/recall between the two.

**Design note:** this notebook's own Jupyter kernel is never assumed to have any project dependency (torch, transformers, pandas, etc.) installed, and nothing is ever installed into it. Every project-specific command runs through a dedicated, fully isolated virtual environment this notebook creates under your home directory (`~/.amazon_sentiment_venv`), using only Python's standard-library `venv` module -- no conda/anaconda needed, and no dependency on write access to whatever kernel environment you happen to be running under. The venv is deliberately isolated from the kernel's own site-packages (not `--system-site-packages`), so `pip install` always installs a correct, working copy of everything (including a CUDA-enabled torch) rather than silently reusing whatever -- possibly broken or CPU-only -- version happens to already be installed in a shared/managed environment. Just run all cells top to bottom.

In [ ]:
# This notebook lives at experiments/bert_class_balancing.ipynb, one
# level below the repo root, and Jupyter starts a notebook's cwd at
# its own directory -- so cwd here is .../experiments, not the repo
# root. Every cell after this one uses paths relative to the repo
# root, so cd up first.
import os

if not os.path.isfile("configs/bert_config.yaml") and os.path.isfile("../configs/bert_config.yaml"):
    %cd ..

# Assumes the repo is already cloned and this notebook is being run
# from inside it (e.g. a persistent GPU machine/cluster you manage
# yourself). Just makes sure you're on the right branch and up to date.
# Once PR #8 (https://github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis/pull/8)
# is merged, change BRANCH to "main".
BRANCH = "feat/bert-class-weighting"

!git checkout {BRANCH}

# A previous run of this notebook may have left configs/bert_config.yaml
# locally modified (see set_class_weights below) -- stash it first so
# `git pull` can't silently fail/conflict against an uncommitted change.
status = !git status --porcelain configs/bert_config.yaml
if status:
    print("Local changes to configs/bert_config.yaml detected -- stashing before pull.")
    !git stash push -- configs/bert_config.yaml

!git pull
!pwd && ls

In [ ]:
# Creates (or reuses) a private virtual environment under your home
# directory, using only the stdlib `venv` module -- no conda/anaconda,
# no --user installs, no write access to the kernel's own environment
# needed. Deliberately NOT using system_site_packages=True: on a
# managed cluster, whatever's already installed in the kernel's own
# environment (e.g. a CPU-only or otherwise broken torch build) would
# get silently reused by pip instead of installing a correct one, which
# is exactly what happened here (`pip show torch` pointed at
# /opt/miniconda/envs/.../site-packages instead of this venv). A fully
# isolated venv guarantees pip installs its own copy of everything.
import os
import shutil
import sys
import venv

VENV_DIR = os.path.expanduser("~/.amazon_sentiment_venv")
VENV_PYTHON = os.path.join(VENV_DIR, "bin", "python")

# If a previous run created this venv with system_site_packages=True
# (the earlier, broken version of this cell), remove it so it gets
# rebuilt fully isolated below.
pyvenv_cfg = os.path.join(VENV_DIR, "pyvenv.cfg")
if os.path.isfile(pyvenv_cfg) and "include-system-site-packages = true" in open(pyvenv_cfg).read().lower():
    print(f"Removing old system-site-packages venv at {VENV_DIR} so it can be rebuilt isolated...")
    shutil.rmtree(VENV_DIR)

if os.path.isfile(VENV_PYTHON):
    print(f"Reusing existing venv at {VENV_DIR}")
else:
    print(f"Creating venv at {VENV_DIR} (based on {sys.executable})...")
    venv.create(VENV_DIR, system_site_packages=False, with_pip=True)
    print("Done.")

print("Using venv Python:", VENV_PYTHON)

In [ ]:
# The default (unpinned) PyPI `torch` wheel bundles whatever CUDA
# runtime is current as of that torch release, which can be newer than
# what this node's NVIDIA driver supports -- exactly what happened here
# (driver reports a max supported CUDA of 12.3 via `nvidia-smi`, but the
# default torch wheel needed something newer, causing "CUDA
# initialization: The NVIDIA driver on your system is too old").
# Installing a build explicitly tagged for CUDA 12.1 first satisfies
# requirements.txt's unpinned torch>=2.0 before it's processed, and
# 12.1 is safely within what a driver supporting up to 12.3 can run.
!{VENV_PYTHON} -m pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!{VENV_PYTHON} -m pip install -q -r requirements.txt

In [ ]:
!{VENV_PYTHON} -c "import torch; print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none -- attach/select a GPU before continuing')"

## 1. Baseline run (`use_class_weights: false`)

This is already the default in `configs/bert_config.yaml`, but the next cell sets it explicitly so this notebook is safe to re-run from the top.

In [ ]:
import re

CONFIG_PATH = "configs/bert_config.yaml"


def set_class_weights(enabled: bool) -> None:
    """Flip use_class_weights via a targeted line replacement rather than
    a full yaml.safe_load/safe_dump round-trip, which would silently
    strip every comment in the file (including the RQ2 explanation right
    above this same setting)."""
    text = open(CONFIG_PATH).read()
    new_text, count = re.subn(
        r"^(\s*use_class_weights:[ \t]*)(true|false)[ \t]*$",
        lambda m: f"{m.group(1)}{'true' if enabled else 'false'}",
        text,
        count=1,
        flags=re.MULTILINE,
    )
    if count == 0:
        raise RuntimeError(
            f"Could not find a 'use_class_weights: true|false' line in {CONFIG_PATH}"
        )
    open(CONFIG_PATH, "w").write(new_text)
    print(f"use_class_weights set to {enabled}")


set_class_weights(False)

In [ ]:
!{VENV_PYTHON} src/train.py

In [ ]:
!{VENV_PYTHON} src/evaluate.py

## 2. Weighted run (`use_class_weights: true`)

In [ ]:
set_class_weights(True)

In [ ]:
!{VENV_PYTHON} src/train.py

In [ ]:
!{VENV_PYTHON} src/evaluate.py

## 3. Compare baseline vs. weighted (RQ2 answer)

Reads the structured JSON results both runs saved to `outputs/` and compares them directly — the neutral-class row is the one RQ2 cares about. A `NaN` for a metric means that class was never predicted at all (undefined), which is distinct from a real `0.0` (predicted, but always wrong). This cell only uses stdlib (`json`), consistent with keeping the notebook's own kernel dependency-free.

In [ ]:
import json

with open("outputs/evaluation_results_baseline.json") as f:
    baseline = json.load(f)

with open("outputs/evaluation_results_weighted.json") as f:
    weighted = json.load(f)

print(f"{'Metric':<24}{'Baseline':>12}{'Weighted':>12}{'Delta':>12}")
print("-" * 60)


def row(label, b, w):
    print(f"{label:<24}{b:>12.4f}{w:>12.4f}{w - b:>+12.4f}")


row("Accuracy", baseline["accuracy"], weighted["accuracy"])
row("Precision Macro", baseline["precision_macro"], weighted["precision_macro"])
row("Recall Macro", baseline["recall_macro"], weighted["recall_macro"])
row("F1 Macro", baseline["f1_macro"], weighted["f1_macro"])
row("F1 Weighted", baseline["f1_weighted"], weighted["f1_weighted"])
print()
for sentiment in ["negative", "neutral", "positive"]:
    row(
        f"Precision ({sentiment})",
        baseline["precision_per_class"][sentiment],
        weighted["precision_per_class"][sentiment],
    )
    row(
        f"Recall ({sentiment})",
        baseline["recall_per_class"][sentiment],
        weighted["recall_per_class"][sentiment],
    )
    row(
        f"F1 ({sentiment})",
        baseline["f1_per_class"][sentiment],
        weighted["f1_per_class"][sentiment],
    )

print()
neutral_f1_delta = weighted["f1_per_class"]["neutral"] - baseline["f1_per_class"]["neutral"]
neutral_recall_delta = weighted["recall_per_class"]["neutral"] - baseline["recall_per_class"]["neutral"]
print(f"Neutral-class F1 delta:     {neutral_f1_delta:+.4f}")
print(f"Neutral-class recall delta: {neutral_recall_delta:+.4f}")
if neutral_f1_delta > 0:
    print(f"RQ2: class weighting IMPROVED neutral-class F1 by {neutral_f1_delta:+.4f}")
elif neutral_f1_delta < 0:
    print(f"RQ2: class weighting HURT neutral-class F1 by {neutral_f1_delta:+.4f}")
else:
    print("RQ2: class weighting made no difference to neutral-class F1")

In [ ]:
# Plain stdlib formatting instead of pandas -- keeps this cell dependency-free too.
labels = ["negative", "neutral", "positive"]


def print_confusion_matrix(title, cm):
    print(title)
    print(" " * 12 + "".join(f"{l:>10}" for l in labels))
    for true_label, row_vals in zip(labels, cm):
        print(f"{true_label:>12}" + "".join(f"{v:>10}" for v in row_vals))


print_confusion_matrix("Baseline confusion matrix (rows=true, cols=predicted):", baseline["confusion_matrix"])
print()
print_confusion_matrix("Weighted confusion matrix (rows=true, cols=predicted):", weighted["confusion_matrix"])

## 4. Commit the results

If you're running this on a persistent machine (your own GPU box, a server/cluster you SSH into, etc.), the repo and its `.git` history are already right here — just commit normally:
```bash
git add outputs/
git commit -m "Add baseline/weighted BERT evaluation results"
git push
```

If instead you're on an ephemeral cloud notebook (Colab/Kaggle) where the filesystem disappears when the session ends, either download `outputs/` via the file browser before disconnecting, or push directly from here:

In [ ]:
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add outputs/
# !git commit -m "Add baseline/weighted BERT evaluation results"
# !git push https://<github-token>@github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis.git {BRANCH}